
# V31 OFFICIAL — Two-Stage Cascaded BanglaBERT

**Target: Kaggle ≥ 0.600**

V30 reached **0.5772** validation, so V31 focuses on the structural bottleneck rather than adding
another generic encoder.

This notebook follows the strongest idea from the 2025 BanglaLP cascaded hate-speech systems:

### Stage 1 — dedicated binary toxicity detector
A BanglaBERT binary classifier is trained on all official training rows.

### Stage 2 — refinement / attribute classifier
An enhanced BanglaBERT with learned attention pooling is trained on:

- all toxic examples
- the hardest non-toxic examples, selected from OOF toxicity probabilities

The second stage predicts:

- refined `is_toxic`
- implicit toxicity
- sarcasm
- profanity
- threatening
- identity attack
- sexual content

So it can correct Stage-1 false positives rather than acting as a rigid hierarchy.

### Ensemble

V31 combines the new cascade with the strongest existing V30/V29 OOF signals:

- V30 regularized stacker
- lexical TF-IDF model
- V31 Stage-1 toxic probability
- V31 Stage-2 refinement/attribute probabilities

All model-family ranking is train-only OOF. Validation is used only for a small final calibration
menu.

Correct official split only: **3292 / 500 / 1000**.

Send `v31_results.zip` to ChatGPT before Kaggle submission.


In [ ]:

# 1. Upload V31 data
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v31_OFFICIAL_data.zip" in uploaded, "Upload v31_OFFICIAL_data.zip."

DATA_DIR = pathlib.Path("/content/v31_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v31_OFFICIAL_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ",p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install + BanglaBERT preflight
import sys, subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn","scipy","pandas","numpy","tqdm","safetensors"
])

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig,AutoTokenizer
from normalizer import normalize

MODEL_NAME="csebuetnlp/banglabert"

_=AutoConfig.from_pretrained(MODEL_NAME)
_=AutoTokenizer.from_pretrained(MODEL_NAME)

print("Normalizer:",normalize("আমি   ভালো আছি।"))
print("✅ BanglaBERT ready.")


In [ ]:

# 3. Imports, strict official split, preprocessing
import gc,re,math,json,random,zipfile,unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from normalizer import normalize

SEED=2026
N_FOLDS=5
MODEL_NAME="csebuetnlp/banglabert"
MAX_LENGTH=128
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
BATCH=8
ACCUM=2
EVAL_BATCH=32
LR=1.5e-5
HEAD_LR=3.0e-5
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.10

LABELS=[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]

RUN_DIR=Path("/content/v31_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

seed_all(SEED)
assert torch.cuda.is_available(),"Enable T4 GPU."

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
sample=pd.read_csv(DATA_DIR/"sample_submission.csv",dtype={"ID":str})

v29=np.load(DATA_DIR/"v29_probabilities.npz")
v30=np.load(DATA_DIR/"v30_new_probabilities.npz")

assert len(train)==3292 and len(valid)==500 and len(test)==1000
assert test.ID.tolist()==sample.ID.tolist()
assert set(train.ID).isdisjoint(valid.ID)
assert set(train.ID).isdisjoint(test.ID)
assert set(valid.ID).isdisjoint(test.ID)

def clean_text(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    x=re.sub(r"\s+"," ",x).strip()
    try:
        x=normalize(x)
    except Exception:
        pass
    return re.sub(r"\s+"," ",x).strip()

def group_key(x):
    x=unicodedata.normalize("NFKC",str(x)).lower()
    return re.sub(r"\s+"," ",x).strip()

for df in [train,valid,test]:
    df["model_text"]=df.text.map(clean_text)
    df["group_key"]=df.text.map(group_key)

print("✅ official split verified")
display(pd.DataFrame({
    "train":train[LABELS].mean(),
    "validation":valid[LABELS].mean()
}).round(4))


In [ ]:

# 4. Recreate the same train-only grouped folds used by V29/V30
groups=train.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos=np.array([train[l].sum() for l in LABELS],float)
total=float(len(train))
rarity=1/np.sqrt(np.maximum(global_pos,1))
mat=groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+0.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fp=np.zeros((N_FOLDS,len(LABELS)))
fs=np.zeros(N_FOLDS)
assign={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])

    costs=[]
    for f in range(N_FOLDS):
        p=fp.copy()
        s=fs.copy()
        p[f]+=vec
        s[f]+=size
        costs.append(
            np.mean((p/np.maximum(global_pos[None,:],1)-1/N_FOLDS)**2)
            +0.30*np.mean((s/total-1/N_FOLDS)**2)
        )

    best=int(np.argmin(costs))
    assign[row["group_key"]]=best
    fp[best]+=vec
    fs[best]+=size

train["fold"]=train.group_key.map(assign).astype(int)

assert train.groupby("group_key").fold.nunique().max()==1
assert set(train.fold)==set(range(5))

print(train.groupby("fold").size().to_dict())
display(train.groupby("fold")[LABELS].sum())


In [ ]:

# 5. Dataset / pooling helpers
class TextDS(Dataset):
    def __init__(self,df,tok,labels=True):
        self.text=df.model_text.astype(str).tolist()
        self.y=df[LABELS].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.text)

    def __getitem__(self,i):
        e=self.tok(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,padding=True,pad_to_multiple_of=8,return_tensors="pt"
        )
        if y is not None:
            b["labels"]=y
        return b

class AttentionPooling(nn.Module):
    def __init__(self,h):
        super().__init__()
        self.score=nn.Linear(h,1)

    def forward(self,hs,mask):
        s=self.score(hs).squeeze(-1)
        s=s.masked_fill(mask==0,-1e4)
        a=torch.softmax(s,dim=1).unsqueeze(-1)
        return (hs*a).sum(1)

class RefineBanglaBERT(nn.Module):
    def __init__(self,name):
        super().__init__()
        self.base=AutoModel.from_pretrained(name)
        h=self.base.config.hidden_size
        self.pool=AttentionPooling(h)

        self.head=nn.Sequential(
            nn.Dropout(0.10),
            nn.Linear(h*2,512),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(512,7),
        )

    def forward(self,input_ids,attention_mask,token_type_ids=None):
        kw={"input_ids":input_ids,"attention_mask":attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"]=token_type_ids

        out=self.base(**kw)
        hs=out.last_hidden_state

        att=self.pool(hs,attention_mask)
        cls=hs[:,0]

        return self.head(torch.cat([cls,att],dim=1))

@torch.no_grad()
def predict_seq(model,loader,custom=False):
    model.eval()
    out=[]

    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp) if custom else model(**inp).logits

        out.append(torch.sigmoid(logits.float()).cpu().numpy())

    return np.concatenate(out)


In [ ]:

# 6. Stage 1 — dedicated binary BanglaBERT
fold_seeds=[2026,17,3407,1337,42]

def train_stage1():
    tok=AutoTokenizer.from_pretrained(MODEL_NAME)

    oof=np.zeros(len(train),np.float32)
    val_parts=[]
    test_parts=[]
    hist=[]

    for fold in range(5):
        tr=train[train.fold!=fold].copy()
        va=train[train.fold==fold].copy()

        cache=RUN_DIR/f"stage1_fold_{fold}.npz"
        hp=RUN_DIR/f"stage1_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]
                pv=z["valid"]
                pe=z["external_valid"]
                pt=z["test"]
            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse stage1",fold)
                oof[idx]=pv
                val_parts.append(pe)
                test_parts.append(pt)
                continue

        seed_all(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        col=Coll(tok)
        tl=DataLoader(
            TextDS(tr,tok,True),batch_size=BATCH,shuffle=True,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        vl=DataLoader(
            TextDS(va,tok,True),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        evl=DataLoader(
            TextDS(valid,tok,False),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        tel=DataLoader(
            TextDS(test,tok,False),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )

        model=AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,num_labels=1
        ).to("cuda")

        opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)

        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")

        sv=[]
        se=[]
        st=[]
        fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)
            run=0.0
            seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"stage1 f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels")[:,0].to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp).logits[:,0]

                loss=torch.nn.functional.binary_cross_entropy_with_logits(
                    logits.float(),y
                )

                scaler.scale(loss/ACCUM).backward()
                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(set_to_none=True)

            rec={"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}
            print(rec)
            fh.append(rec)
            hist.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(predict_seq(model,vl)[:,0])
                se.append(predict_seq(model,evl)[:,0])
                st.append(predict_seq(model,tel)[:,0])

        pv=np.mean(sv,axis=0)
        pe=np.mean(se,axis=0)
        pt=np.mean(st,axis=0)

        oof[va.index.to_numpy()]=pv
        val_parts.append(pe)
        test_parts.append(pt)

        np.savez_compressed(
            cache,idx=va.index.to_numpy(),valid=pv,
            external_valid=pe,test=pt
        )

        del model,opt,sched,scaler,tl,vl,evl,tel
        gc.collect()
        torch.cuda.empty_cache()

    pval=np.mean(val_parts,axis=0)
    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/"stage1_oof_valid_test.npz",
        oof=oof,validation=pval,test=ptest
    )
    pd.DataFrame(hist).to_csv(
        RUN_DIR/"stage1_training_history.csv",index=False
    )

    return oof,pval,ptest

s1_oof,s1_val,s1_test=train_stage1()
print("✅ Stage 1 complete.")


In [ ]:

# 7. Build fold-safe hard-negative sets for Stage 2
# V30 stack toxicity probability is OOF for training rows.
v30_toxic_oof=v30["stack_oof"][:,0]

hard_negative=np.zeros(len(train),dtype=bool)

for fold in range(5):
    pool=(train.fold!=fold) & (train.is_toxic==0)
    idx=np.where(pool)[0]

    # top 35% highest-risk non-toxic rows from OOF Stage-1 ensemble
    k=max(1,int(round(0.35*len(idx))))
    chosen=idx[np.argsort(-v30_toxic_oof[idx])[:k]]
    hard_negative[chosen]=True

print("Hard non-toxic rows selected:",int(hard_negative.sum()))
print("All toxic rows:",int(train.is_toxic.sum()))


In [ ]:

# 8. Stage 2 — attention-pooling refinement model
def refine_loss(logits,y,frame):
    raw=torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),y,reduction="none"
    )

    # Refine toxicity + six attributes.
    pos=frame[LABELS].sum().to_numpy(float)
    neg=len(frame)-pos
    boost=(np.maximum(neg,1)/np.maximum(pos,1))**0.20
    boost=np.clip(boost,1.0,2.0)
    boost[0]=1.0

    pb=torch.tensor(boost,dtype=torch.float32,device=logits.device).unsqueeze(0)
    w=torch.where(y>0.5,pb,torch.ones_like(y))

    # Competition-shaped task weights.
    task=torch.tensor(
        [0.50]+[0.50/6]*6,
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)

    return (raw*w*task).sum(1).mean()

def train_stage2():
    tok=AutoTokenizer.from_pretrained(MODEL_NAME)

    oof=np.zeros((len(train),7),np.float32)
    val_parts=[]
    test_parts=[]
    hist=[]

    for fold in range(5):
        train_mask=(train.fold!=fold)

        # all toxic + hard non-toxic
        use=train_mask & (
            (train.is_toxic==1)
            | hard_negative
        )

        tr=train[use].copy()
        va=train[train.fold==fold].copy()

        cache=RUN_DIR/f"stage2_fold_{fold}.npz"
        hp=RUN_DIR/f"stage2_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]
                pv=z["valid"]
                pe=z["external_valid"]
                pt=z["test"]
            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse stage2",fold)
                oof[idx]=pv
                val_parts.append(pe)
                test_parts.append(pt)
                continue

        print(
            f"stage2 fold {fold} | train={len(tr)} "
            f"toxic={int(tr.is_toxic.sum())} "
            f"hard_non_toxic={int((tr.is_toxic==0).sum())}"
        )

        seed_all(fold_seeds[fold]+100)
        gc.collect()
        torch.cuda.empty_cache()

        col=Coll(tok)
        tl=DataLoader(
            TextDS(tr,tok,True),batch_size=BATCH,shuffle=True,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        vl=DataLoader(
            TextDS(va,tok,True),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        evl=DataLoader(
            TextDS(valid,tok,False),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        tel=DataLoader(
            TextDS(test,tok,False),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )

        model=RefineBanglaBERT(MODEL_NAME).to("cuda")

        backbone=[]
        head=[]

        for n,p in model.named_parameters():
            if n.startswith("base."):
                backbone.append(p)
            else:
                head.append(p)

        opt=torch.optim.AdamW([
            {"params":backbone,"lr":LR,"weight_decay":WEIGHT_DECAY},
            {"params":head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
        ])

        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")

        sv=[]
        se=[]
        st=[]
        fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)
            run=0.0
            seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"stage2 f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp)

                loss=refine_loss(logits,y,tr)

                scaler.scale(loss/ACCUM).backward()
                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(set_to_none=True)

            rec={"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}
            print(rec)
            fh.append(rec)
            hist.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(predict_seq(model,vl,custom=True))
                se.append(predict_seq(model,evl,custom=True))
                st.append(predict_seq(model,tel,custom=True))

        pv=np.mean(sv,axis=0)
        pe=np.mean(se,axis=0)
        pt=np.mean(st,axis=0)

        oof[va.index.to_numpy()]=pv
        val_parts.append(pe)
        test_parts.append(pt)

        np.savez_compressed(
            cache,idx=va.index.to_numpy(),valid=pv,
            external_valid=pe,test=pt
        )

        del model,opt,sched,scaler,tl,vl,evl,tel
        gc.collect()
        torch.cuda.empty_cache()

    pval=np.mean(val_parts,axis=0)
    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/"stage2_oof_valid_test.npz",
        oof=oof,validation=pval,test=ptest
    )
    pd.DataFrame(hist).to_csv(
        RUN_DIR/"stage2_training_history.csv",index=False
    )

    return oof,pval,ptest

s2_oof,s2_val,s2_test=train_stage2()
print("✅ Stage 2 complete.")


In [ ]:

# 9. Competition metrics / calibration
y_train=train[LABELS].to_numpy(np.int8)
y_valid=valid[LABELS].to_numpy(np.int8)
folds=train.fold.to_numpy(int)

def f1_pos(y,p):
    y=np.asarray(y,int); p=np.asarray(p,int)
    tp=((y==1)&(p==1)).sum()
    fp=((y==0)&(p==1)).sum()
    fn=((y==1)&(p==0)).sum()
    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.0

def f1_macro(y,p):
    return float(np.mean([
        f1_pos((y==c).astype(int),(p==c).astype(int))
        for c in [0,1]
    ]))

def metric(label,y,p):
    return f1_macro(y,p) if label=="is_toxic" else f1_pos(y,p)

def official(y,p):
    scores=[metric(l,y[:,j],p[:,j]) for j,l in enumerate(LABELS)]
    return float(.5*scores[0]+.5*np.mean(scores[1:])),scores

def topk(p,rate):
    k=int(round(rate*len(p)))
    k=max(0,min(k,len(p)))
    out=np.zeros(len(p),np.int8)
    if k:
        out[np.argsort(-p)[:k]]=1
    return out

def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))


In [ ]:

# 10. Train-only candidate ranking, then tiny validation calibration menu

# Existing strong signals:
lex_oof=v29["lexical_oof"]
lex_val=v29["lexical_validation"]
lex_test=v29["lexical_test"]

stack_oof=v30["stack_oof"]
stack_val=v30["stack_validation"]
stack_test=v30["stack_test"]

def candidate_probs(j,split):
    if split=="train":
        s1,s2,stack,lex=s1_oof,s2_oof,stack_oof,lex_oof
    elif split=="validation":
        s1,s2,stack,lex=s1_val,s2_val,stack_val,lex_val
    else:
        s1,s2,stack,lex=s1_test,s2_test,stack_test,lex_test

    if j==0:
        return {
            "stage1":s1,
            "stage2":s2[:,0],
            "cascade_7030":.70*s1+.30*s2[:,0],
            "cascade_stack":.45*s1+.25*s2[:,0]+.30*stack[:,0],
            "stack":stack[:,0],
            "lex":lex[:,0],
        }

    return {
        "stage2":s2[:,j],
        "stage2_stack":.70*s2[:,j]+.30*stack[:,j],
        "stage2_lex":.70*s2[:,j]+.30*lex[:,j],
        "cascade_semantic":.60*s2[:,j]+.25*stack[:,j]+.15*lex[:,j],
        "stack":stack[:,j],
        "lex":lex[:,j],
    }

# Rank candidates on train OOF using prevalence top-k only.
top_train={}

for j,label in enumerate(LABELS):
    options=[]

    for name,p in candidate_probs(j,"train").items():
        for scale in [0.85,0.95,1.00,1.05,1.15]:
            pred=topk(p,float(train[label].mean())*scale)
            score=metric(label,y_train[:,j],pred)
            options.append({
                "name":name,
                "scale":scale,
                "train_score":score,
                "train_pred_rate":float(pred.mean()),
            })

    options=sorted(
        options,
        key=lambda r:r["train_score"],
        reverse=True
    )

    # keep top 4 distinct family/scale candidates
    top_train[label]=options[:4]

print("Top train-only candidates:")
for l in LABELS:
    print(l,top_train[l])


In [ ]:

# 11. Choose among only top-4 train candidates on validation
selected={}
rows=[]

combined_prior=pd.concat(
    [train[LABELS],valid[LABELS]],ignore_index=True
).mean()

for j,label in enumerate(LABELS):
    best=None

    for rec in top_train[label]:
        name=rec["name"]
        scale=rec["scale"]

        pv=candidate_probs(j,"validation")[name]
        pt=candidate_probs(j,"test")[name]

        vpred=topk(
            pv,
            float(train[label].mean())*scale
        )
        tpred=topk(
            pt,
            float(combined_prior[label])*scale
        )

        vs=metric(label,y_valid[:,j],vpred)
        gap=abs(float(vpred.mean())-float(y_valid[:,j].mean()))

        row={
            **rec,
            "label":label,
            "validation_score":float(vs),
            "validation_pred_rate":float(vpred.mean()),
            "validation_true_rate":float(y_valid[:,j].mean()),
            "robust":float(vs-0.08*gap),
        }
        rows.append(row)

        key=(row["robust"],row["validation_score"],row["train_score"])

        if best is None or key>best["key"]:
            best={
                "key":key,
                "rec":row,
                "validation_pred":vpred,
                "test_pred":tpred,
            }

    selected[label]=best

pd.DataFrame(rows).to_csv(
    RUN_DIR/"v31_validation_selection.csv",index=False
)

print("Selected:")
display(pd.DataFrame([selected[l]["rec"] for l in LABELS]).round(5))


In [ ]:

# 12. Final score / submission
val_pred=np.column_stack(
    [selected[l]["validation_pred"] for l in LABELS]
)
test_pred=np.column_stack(
    [selected[l]["test_pred"] for l in LABELS]
)

vscore,scores=official(y_valid,val_pred)

report=pd.DataFrame({
    "label":LABELS,
    "validation_score":scores,
    "true_rate":y_valid.mean(0),
    "pred_rate":val_pred.mean(0),
    "test_pred_rate":test_pred.mean(0),
})
report.to_csv(
    RUN_DIR/"v31_final_validation_report.csv",index=False
)

SUBMIT_CANDIDATE=bool(
    vscore>=0.600
    and scores[0]>=0.70
)

summary={
    "version":"v31_two_stage_cascade",
    "target":0.600,
    "validation_official_score":float(vscore),
    "toxic_macro_f1":float(scores[0]),
    "attribute_mean_f1":float(np.mean(scores[1:])),
    "SUBMIT_CANDIDATE":SUBMIT_CANDIDATE,
}

(RUN_DIR/"v31_summary.json").write_text(
    json.dumps(summary,indent=2)
)

sub=sample.copy()
for j,l in enumerate(LABELS):
    sub[l]=test_pred[:,j]

assert sub.ID.tolist()==test.ID.tolist()
assert sub[LABELS].isin([0,1]).all().all()

sub.to_csv(
    RUN_DIR/"submission_v31_target600.csv",index=False
)

print(json.dumps(summary,indent=2))
display(report.round(5))

if SUBMIT_CANDIDATE:
    print("\n✅ V31 PASSED 0.600 OFFLINE GATE.")
else:
    print("\n❌ V31 did not pass. Send results ZIP; do not submit yet.")


In [ ]:

# 13. Package results
np.savez_compressed(
    RUN_DIR/"v31_probabilities.npz",
    stage1_oof=s1_oof,stage1_validation=s1_val,stage1_test=s1_test,
    stage2_oof=s2_oof,stage2_validation=s2_val,stage2_test=s2_test,
)

zip_path=Path("/content/v31_results.zip")

with zipfile.ZipFile(
    zip_path,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
